## Setup

Add the project root and import the loss classes used in this notebook.


In [ ]:
import sys
sys.path.insert(0, "../../../src")

import numpy as np
from didgelab import (
    CompositeTairuaLoss,
    FrequencyTuningLoss,
    ScaleTuningLoss,
    PeakQuantityLoss,
    PeakAmplitudeLoss,
    QFactorLoss,
    ModalDensityLoss,
    IntegerHarmonicLoss,
    NearIntegerLoss,
    StretchedOddLoss,
    HighInharmonicLoss,
    HarmonicSplittingLoss,
    note_to_freq,
)


# Scale Tuning Loss

**Purpose:** Pull all detected resonances toward the nearest note of a given **musical scale**. Good for instruments that should "toot" in a specific key (e.g. D minor, blues scale).

**Formula:**

$$L_{scale} = w \cdot \frac{1}{600N} \sum_{i=1}^{N} \alpha_i \min \bigl| 1200 \cdot (\log_2 f_{peak,i} - \log_2 F_{scale}) \bigr|, \quad \alpha_i \propto f_i^{1-p}$$

with \(\alpha_i\) renormalized so \(\sum_i \alpha_i = N\).

**Symbols:**
- $ F_{scale} $: Allowed log2 frequencies (scale notes from base + intervals).
- $ f_{peak,i} $: Detected resonance frequency.
- $ \min $: Distance in cents to the closest in-tune note.
- $ \alpha_i $: Per-peak weight \(\propto f_i^{1-p}\), sum preserved (\(N\)).
- $ p $: `favor_lower_frequencies` — **1 = neutral** (uniform); **>1** favors lower frequencies (e.g. \(p=2\) → \(\alpha \propto 1/f\)); **<1** favors higher frequencies.
- 600: Normalization constant.
- $ w $: Weight for scale adherence.

Changing \(p\) redistributes weight across peaks without changing the overall scale-loss budget.

In [ ]:
# base_note: MIDI note number of scale root (e.g. 60 = C4)
# intervals: semitone steps from root, e.g. [0, 2, 4, 5, 7, 9, 11] for major
# favor_lower_frequencies: 1.0 = neutral; >1 favors lower peaks; <1 favors higher
scale_component = ScaleTuningLoss(
    base_note=60,  # C4
    intervals=[0, 2, 4, 5, 7, 9, 11],  # major scale
    weight=10.0,
    favor_lower_frequencies=2.0,  # prioritize drone / low toots (default is 1.0)
)
# loss.add_component("scale", scale_component)